In [ ]:
import os
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')

from fastai.vision.all import load_learner, PILImage
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
import io

# Works from repo root (Binder) or notebooks/ dir (local)
MODEL_PATH = Path('models/sport_classifier.pkl')
if not MODEL_PATH.exists():
    MODEL_PATH = Path('../models/sport_classifier.pkl')

learn = load_learner(MODEL_PATH)
print(f"✅ Model loaded — classes: {learn.dls.vocab}")

In [ ]:
title = widgets.HTML(
    '<h2 style="font-family:sans-serif;margin-bottom:4px">🏀⚽ Sport Classifier</h2>'
    '<p style="font-family:sans-serif;color:#555;margin-top:0">Upload a photo — the model will predict <b>basketball</b> or <b>soccer</b>.</p>'
)
upload = widgets.FileUpload(accept='image/*', multiple=False, description='Upload image')
out    = widgets.Output()

def classify(change):
    with out:
        clear_output(wait=True)
        if not upload.value:
            return
        val     = upload.value
        # ipywidgets ≥8 gives a tuple, <8 gives a dict
        content = val[0]['content'] if isinstance(val, tuple) else list(val.values())[0]['content']

        img = PILImage.create(io.BytesIO(bytes(content)))
        pred, pred_idx, probs = learn.predict(img)
        conf  = float(probs[pred_idx]) * 100
        emoji = '🏀' if str(pred) == 'basketball' else '⚽'

        display(widgets.Image(value=bytes(content), format='jpeg', width=320))
        display(widgets.HTML(
            f'<h3 style="font-family:sans-serif">{emoji} {str(pred).capitalize()}'
            f' <span style="color:#888;font-size:0.85em">({conf:.1f}% confidence)</span></h3>'
        ))

        for cls, p in zip(learn.dls.vocab, probs):
            pct   = float(p) * 100
            bar_w = int(pct * 2.5)
            color = '#2ecc71' if cls == str(pred) else '#bdc3c7'
            display(widgets.HTML(
                f'<div style="font-family:sans-serif;margin:4px 0">'
                f'<span style="display:inline-block;width:130px">{cls}</span>'
                f'<span style="display:inline-block;width:{bar_w}px;height:16px;'
                f'background:{color};vertical-align:middle;border-radius:3px"></span>'
                f'&nbsp;{pct:.1f}%</div>'
            ))

upload.observe(classify, names='value')
display(widgets.VBox([title, upload, out]))

In [ ]:
title   = widgets.HTML('<h2 style="font-family:sans-serif">🏀⚽ Sport Classifier</h2>'
                       '<p style="font-family:sans-serif;color:#555">Upload a photo and the model will predict: <b>basketball</b> or <b>soccer</b>.</p>')
upload  = widgets.FileUpload(accept='image/*', multiple=False, description='Upload image')
out     = widgets.Output()

def classify(change):
    with out:
        clear_output(wait=True)
        if not upload.value:
            return
        # ipywidgets >= 8 returns a tuple; < 8 returns a dict
        val = upload.value
        content = val[0]['content'] if isinstance(val, tuple) else list(val.values())[0]['content']

        img = PILImage.create(io.BytesIO(bytes(content)))
        pred, pred_idx, probs = learn.predict(img)
        conf = float(probs[pred_idx]) * 100

        # Show the image
        display(widgets.Image(value=bytes(content), format='jpeg', width=320))

        # Result header
        emoji = '🏀' if str(pred) == 'basketball' else '⚽'
        display(widgets.HTML(f'<h3 style="font-family:sans-serif">'
                              f'{emoji} {str(pred).capitalize()} &nbsp;'
                              f'<span style="color:#888;font-size:0.85em">({conf:.1f}% confidence)</span>'
                              f'</h3>'))

        # Probability bars
        for cls, p in zip(learn.dls.vocab, probs):
            pct = float(p) * 100
            bar_w = int(pct * 2.5)
            color = '#2ecc71' if cls == str(pred) else '#bdc3c7'
            display(widgets.HTML(
                f'<div style="font-family:sans-serif;margin:4px 0">'
                f'<span style="display:inline-block;width:120px">{cls}</span>'
                f'<span style="display:inline-block;width:{bar_w}px;height:16px;'
                f'background:{color};vertical-align:middle;border-radius:3px"></span>'
                f'&nbsp;{pct:.1f}%</div>'
            ))

upload.observe(classify, names='value')
display(widgets.VBox([title, upload, out]))